# Packages

In [1]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 --index-url https://download.pytorch.org/whl/cu121

Found existing installation: torch 2.11.0+cpu
Uninstalling torch-2.11.0+cpu:
  Successfully uninstalled torch-2.11.0+cpu
Found existing installation: torchvision 0.26.0+cpu
Uninstalling torchvision-0.26.0+cpu:
  Successfully uninstalled torchvision-0.26.0+cpu
Found existing installation: torchaudio 2.11.0+cpu
Uninstalling torchaudio-2.11.0+cpu:
  Successfully uninstalled torchaudio-2.11.0+cpu
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.9/780.9 MB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 19.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 44.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 40.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 19.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 33.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.7

In [8]:
import pandas as pd             #only for testing ?
import nltk
import zipfile                  #only for testing
from google.colab import drive #only for testing
from datasets import Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, recall_score, f1_score

# Import Data

In [3]:
drive.mount('/content/gdrive')

zip_path = "/content/gdrive/MyDrive/liar_project/raw_data.zip"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('/content/dataset')

Mounted at /content/gdrive


# Data Preprocessing variables for the test dataset

In [4]:
def simplify_label(label):
    if label in ["true", "mostly-true"]:
        return "trustworthy"

    elif label == "half-true":
        return "questionable"

    else:
        return "unreliable"

In [5]:
label2id = {"trustworthy": 0, "questionable": 1, "unreliable": 2}
id2label = {v: k for k, v in label2id.items()}

# load test data and preprocess it to have the correct structure for the model

In [23]:
path = "/content/dataset/raw_data/liar/test.tsv"
df_test = pd.read_csv(path , sep='\t', header = 0)
df_test.columns = ['id', 'label', 'statement', 'subject','speaker', "job_title", 'state', 'party', 'barely_true_counts', 'false_counts',
'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts','context']
df_test["simple_label"] = df_test["label"].apply(simplify_label)
df_test["labels"] = df_test["simple_label"].map(label2id)
test_ds = Dataset.from_pandas(df_test)

# load trained tokenizer and Model

In [24]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
model_path = "/content/drive/MyDrive/model_roberta_3classes" # paht to the model, either local or GCP

In [26]:

tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True) # Tokenizer is defined in in the model_path config file, pulled automatically here
model = AutoModelForSequenceClassification.from_pretrained(model_path, local_files_only=True) # modeltype and weights are automatically imported from the model_path files
model.to("cpu")   # oder "cuda" for GPU
model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

# tokenize data

In [27]:
max_length = 256

def tokenize_batch(batch):
    return tokenizer(
        batch["statement"],      # here is the colum defined from the test dataset - could be text as string
        padding="max_length",
        truncation=True,
        max_length=max_length
    )

test_tokenized = test_ds.map(tokenize_batch, batched=True)

Map:   0%|          | 0/1266 [00:00<?, ? examples/s]

In [28]:
needed = {"input_ids", "attention_mask", "labels"}

test_tokenized = test_tokenized.remove_columns([c for c in test_tokenized.column_names if c not in needed]) # only needed because our test dataset has more columns than just the statement

In [29]:
test_tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [30]:

test_loader = DataLoader(test_tokenized, batch_size=16)

# predict

In [31]:
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels = batch["labels"]

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# calculate metrices

In [32]:


accuracy = accuracy_score(all_labels, all_preds)
recall_macro = recall_score(all_labels, all_preds, average="macro")
f1_macro = f1_score(all_labels, all_preds, average="macro")

print("Accuracy:", accuracy)
print("Recall (macro):", recall_macro)
print("F1 (macro):", f1_macro)

Accuracy: 0.3578199052132701
Recall (macro): 0.33465189873417717
F1 (macro): 0.19353496794401212


# Prediction of a statement

In [33]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_path = "/content/drive/MyDrive/model_roberta_3classes"

tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(model_path, local_files_only=True)
model.eval()

def predict_statement(text: str):
    # 1. Tokenisieren
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )

    # 2. Modellvorhersage
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    # 3. Wahrscheinlichkeiten berechnen
    probs = F.softmax(logits, dim=1)[0]

    # 4. Klasse bestimmen
    pred_class_id = torch.argmax(probs).item()
    pred_class_label = model.config.id2label[pred_class_id]

    # 5. Wahrscheinlichkeiten in Python-Format
    probs_dict = {
        model.config.id2label[i]: float(probs[i])
        for i in range(len(probs))
    }

    return {
        "text": text,
        "predicted_class": pred_class_label,
        "probabilities": probs_dict
    }


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [34]:
result = predict_statement("The economy is doing great.")
print(result)


{'text': 'The economy is doing great.', 'predicted_class': 'unreliable', 'probabilities': {'trustworthy': 0.2965378761291504, 'questionable': 0.284930020570755, 'unreliable': 0.4185321033000946}}
